In [ ]:
import pickle
import numpy as np
from matplotlib.pyplot import figure
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
with open("../input/smart-reply-sentence-similarity/target_texts.pickle", 'rb') as handle:
    target_texts = pickle.load(handle)

In [ ]:
tqdm.pandas()
df = pd.read_csv("../input/smart-reply-sentence-similarity/target_similarity_matrix.txt", header=None)
df.head()

In [ ]:
# first and last column data contains [ and ]
df.iloc[:,0]

In [ ]:
df.iloc[:,-1:]

In [ ]:
# cleaning first and last columns
df.iloc[:,0] = df.iloc[:,0].str.replace("[", "").str.strip()
df.iloc[:,-1:] = df.iloc[:,-1].str.replace("]", "").str.strip()

In [ ]:
df.iloc[:,0]

In [ ]:
df.iloc[:,-1:]

In [ ]:
# converting datatype to float
target_similarity_matrix = df.astype(float)
target_similarity_matrix.describe()

In [ ]:
# finding 10th neighbour distance to sentence 1
target_similarity_matrix.iloc[1].nsmallest(10).iloc[10-1]

In [ ]:
# extending logic to all rows
def sort_and_select_10_neigh_dist(row):
    return row.nsmallest(10).iloc[10-1]

neighbor_10_dist = target_similarity_matrix.apply(sort_and_select_10_neigh_dist, axis = 1) 

In [ ]:
neighbor_10_dist.describe()

In [ ]:
for x in list(target_similarity_matrix.nsmallest(10, 1).index.astype(int)):
    print(target_texts[x], " - Similarity Score :", target_similarity_matrix[x][1]) 

In [ ]:
# using 40th percentile value as appropriate distance for 10th neighbour
eps_10 = neighbor_10_dist.quantile(0.99)
print(eps_10)

In [ ]:
eps_nums = list()
eps_num_clusters = list()
eps_num_noise = list()
trial_eps = np.linspace(0, neighbor_10_dist.quantile(1.0), num=200)

for eps_num in trial_eps:
    
    if eps_num > 0:    
        print("Taking EPS as", eps_num)
        # Compute DBSCAN
        db = DBSCAN(eps=eps_num, min_samples=2, metric="precomputed", n_jobs=4).fit(target_similarity_matrix)
        #core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
        #core_samples_mask[db.core_sample_indices_] = True
        labels = db.labels_

        # Number of clusters in labels, ignoring noise if present.
        n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise_ = list(labels).count(-1)

        eps_nums.append(eps_num)
        eps_num_clusters.append(n_clusters_)
        eps_num_noise.append(n_noise_)

        print('Estimated number of clusters: %d' % n_clusters_)
        print('Estimated number of noise points: %d' % n_noise_)
        print("-----------")
        
        if n_clusters_ == 1:
            print("Stopping..")
            # least number of clusters
            break

In [ ]:
#figure(num=None, figsize=(15, 6), dpi=80)
fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.set_xlabel('time (s)')
ax1.set_ylabel('exp', color=color)
ax1.plot(eps_nums, eps_num_clusters, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()

ax2.plot(eps_nums, eps_num_noise)
plt.show()

In [ ]:
eps_num = 0.0037917503759489577
print("Taking EPS as", eps_num)
# Compute DBSCAN
db = DBSCAN(eps=eps_num, min_samples=2, metric="precomputed", n_jobs=4).fit(target_similarity_matrix)
core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True
labels = db.labels_

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print('Estimated number of clusters: %d' % n_clusters_)
print('Estimated number of noise points: %d' % n_noise_)

In [ ]:

from collections import Counter

for unique_label in set(labels):
    
    class_member_mask = (labels == unique_label)
    print("In cluster", unique_label, "found", Counter(class_member_mask)[True], "points")
    print("Samples")
    print(np.array(target_texts)[class_member_mask])
    print("-------------------------------------")
    #print(Counter(class_member_mask).keys()) # equals to list(set(words))
    #Counter(words).values() # counts the elements' frequency


In [ ]:
with open('target_dbscan.pickle', 'wb') as handle:
    pickle.dump(db, handle, protocol=pickle.HIGHEST_PROTOCOL)